In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import cv2

In [19]:
import cv2
import os
import pandas as pd

def load_images_to_dataframe(image_dir):

    data = []


    for file_name in os.listdir(image_dir):
        file_path = os.path.join(image_dir, file_name)
        

        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):

            img = cv2.imread(file_path)
            

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            

            img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)
            

            alpha = 1.2 
            beta = 20   
            img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)


            image_id = int(os.path.splitext(file_name)[0])
            
            # Append image data and file name to the list
            data.append({
                'image_id': image_id,
                'image_data': img
            })


    df = pd.DataFrame(data)
    return df


In [20]:
df_train = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/train/')
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
df_dev = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/dev')

In [21]:
train_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/train/train.csv')
dev_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/dev/dev.csv')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')

train = pd.merge(train_csv, df_train, on='image_id', how='inner')
dev = pd.merge(dev_csv, df_dev, on='image_id', how='inner')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')


In [22]:
pip install git+https://github.com/indic-transliteration/indic_transliteration_py/@master -U

  Cloning https://github.com/indic-transliteration/indic_transliteration_py/ (to revision master) to /tmp/pip-req-build-3b6bjz6c
  Running command git clone --filter=blob:none --quiet https://github.com/indic-transliteration/indic_transliteration_py/ /tmp/pip-req-build-3b6bjz6c
  Resolved https://github.com/indic-transliteration/indic_transliteration_py/ to commit c2142c61166a07c36a311aef59a516ea5cf61910
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


In [23]:
def text_preprocessing(text):
    import re
    pattern = re.compile('[@#\/]\S+')
    text = pattern.sub(r'',text)

    pattern = re.compile('\d+')
    text = pattern.sub(r'', text)

    pattern = re.compile(r'https?:\/\/\S+|www\.\S+|ftp:\/\/\S+|mailto:\S+|https?:')

    # First remove URLs
    text = pattern.sub('', text)

    # Remove newline characters (\n) and carriage returns (\r)
    text = text.replace('\n', ' ').replace('\r', '')

    # Remove extra spaces (including multiple spaces)
    text = re.sub(r'\s+', ' ', text).strip()

    import string
    punc = string.punctuation

    text = text.translate(str.maketrans('','',punc))

    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F300-\U0001F5FF"  # Symbols & Pictographs
        "\U0001F680-\U0001F6FF"  # Transport & Map Symbols
        "\U0001F1E0-\U0001F1FF"  # Flags (iOS)
        "\U00002700-\U000027BF"  # Dingbats
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        "\U00002600-\U000026FF"  # Miscellaneous Symbols
        "\U00002B50-\U00002B55"  # Stars and other symbols
        "]+",
        flags=re.UNICODE
    )

    text = emoji_pattern.sub(r'', text)

    tamil_stopwords = [
    "ஒரு", "என்று", "மற்றும்", "இந்த", "இது", "என்ற", "கொண்டு", "என்பது", "பல", "ஆகும்",
    "அல்லது", "அவர்", "நான்", "உள்ள", "அந்த", "இவர்", "என", "முதல்", "என்ன", "இருந்து",
    "சில", "என்", "போன்ற", "வேண்டும்", "வந்து", "இதன்", "அது", "அவன்", "தான்", "பலரும்",
    "என்னும்", "மேலும்", "பின்னர்", "கொண்ட", "இருக்கும்", "தனது", "உள்ளது", "போது", "என்றும்",
    "அதன்", "தன்", "பிறகு", "அவர்கள்", "வரை", "அவள்", "நீ", "ஆகிய", "இருந்தது", "உள்ளன",
    "வந்த", "இருந்த", "மிகவும்", "இங்கு", "மீது", "ஓர்", "இவை", "இந்தக்", "பற்றி", "வரும்",
    "வேறு", "இரு", "இதில்", "போல்", "இப்போது", "அவரது", "மட்டும்", "இந்தப்", "எனும்", "மேல்",
    "பின்", "சேர்ந்த", "ஆகியோர்", "எனக்கு", "இன்னும்", "அந்தப்", "அன்று", "ஒரே", "மிக", "அங்கு",
    "பல்வேறு", "விட்டு", "பெரும்", "அதை", "பற்றிய", "உன்", "அதிக", "அந்தக்", "பேர்", "இதனால்",
    "அவை", "அதே", "ஏன்", "முறை", "யார்", "என்பதை", "எல்லாம்", "மட்டுமே", "இங்கே", "அங்கே",
    "இடம்", "இடத்தில்", "அதில்", "நாம்", "அதற்கு", "எனவே", "பிற", "சிறு", "மற்ற", "விட", "எந்த",
    "எனவும்", "எனப்படும்", "எனினும்", "அடுத்த", "இதனை", "இதை", "கொள்ள", "இந்தத்", "இதற்கு",
    "அதனால்", "தவிர", "போல", "வரையில்", "சற்று", "எனக்"
    ]

    text_ls = text.split()
    filtered_words = [word for word in text_ls if word not in tamil_stopwords]
    # Join the remaining words back into a string
    text = " ".join(filtered_words)

    from indic_transliteration import sanscript
    from indic_transliteration.sanscript import SchemeMap, SCHEMES, transliterate
    
    
    text = transliterate(text, sanscript.HK, sanscript.TAMIL)


    
    return text

In [24]:
train['transcriptions'] = train['transcriptions'].apply(text_preprocessing)
test['transcriptions'] = test['transcriptions'].apply(text_preprocessing)
dev['transcriptions'] = dev['transcriptions'].apply(text_preprocessing)

In [25]:
# !pip install googletrans==4.0.0-rc1

In [26]:
pip install deep-translator


Note: you may need to restart the kernel to use updated packages.


In [27]:
from PIL import Image
from torchvision import transforms
from deep_translator import GoogleTranslator
import numpy as np
import pandas as pd

augmented_data = []
cnt = 0
img_augmentations = transforms.Compose([
    transforms.ColorJitter(brightness=0.5),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomPosterize(bits=4),
    # transforms.GaussianBlur(kernel_size=3)
])

for idx, row in train.iterrows():
    image_data = row['image_data']
    text = row['transcriptions']
    label = row['labels']
    image_id = row['image_id']

    if label == 1:
        cnt += 1
        print(cnt)


        image = Image.fromarray(image_data.astype('uint8'))
        img_aug = img_augmentations(image)
        img_aug = np.array(img_aug)

 
        text_aug = text
        try:
            translated_to_en = GoogleTranslator(source='ta', target='en').translate(text_aug)
            text_aug = GoogleTranslator(source='en', target='ta').translate(translated_to_en)
        except Exception as e:
            print(f"Translation error (Tamil -> English -> Tamil): {e}")
            print(text)
        
        augmented_data.append({
            'image_data': img_aug,
            'labels': label,
            'image_id': image_id,
            'transcriptions': text_aug
        })


        image = Image.fromarray(image_data.astype('uint8'))
        img_aug = img_augmentations(image)
        img_aug = np.array(img_aug)


        text_aug = text
        try:
            translated_to_ml = GoogleTranslator(source='ta', target='ml').translate(text_aug)
            text_aug = GoogleTranslator(source='ml', target='ta').translate(translated_to_ml)
        except Exception as e:
            print(f"Translation error (Tamil -> Malayalam -> Tamil): {e}")
            print(text)
        
        augmented_data.append({
            'image_data': img_aug,
            'labels': label,
            'image_id': image_id,
            'transcriptions': text_aug
        })


augmented_df = pd.DataFrame(augmented_data)

train = pd.concat([train, augmented_df], ignore_index=True)


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [28]:
train.drop(columns='image_id', inplace=True)
dev.drop(columns='image_id', inplace=True)
test.drop(columns='image_id', inplace=True)

# mBert

In [29]:
import torch
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

class TextDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']
        label = self.df.loc[idx, 'labels']


        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')

        return encoding, torch.tensor(label)

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
bert_model = BertModel.from_pretrained('bert-base-multilingual-cased')

class TextClassifier(nn.Module):
    def __init__(self, bert_model, num_classes):
        super(TextClassifier, self).__init__()
        self.bert_model = bert_model
        self.fc = nn.Linear(768, num_classes)  # mBERT has a hidden size of 768

    def forward(self, input_ids, attention_mask):

        bert_output = self.bert_model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        bert_pool = torch.mean(bert_output, 1)
        output = self.fc(bert_pool)
        return output


train_dataset = TextDataset(train, tokenizer)
dev_dataset = TextDataset(dev, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16)


num_classes = len(train['labels'].unique())
model = TextClassifier(bert_model, num_classes)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in range(5): 
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        labels = batch[1].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Evaluation
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for batch in dev_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        labels = batch[1].to(device)

        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}%")


Epoch 1, Loss: 0.6688810818106214
Epoch 2, Loss: 0.7065671771486229
Epoch 3, Loss: 0.7031848330364049
Epoch 4, Loss: 0.6972247830061155
Epoch 5, Loss: 0.6965065982854255
Accuracy: 26.056338028169016%


In [30]:
from sklearn.metrics import classification_report, f1_score
import numpy as np


model.eval()
with torch.no_grad():
    y_true = []
    y_pred = []
    
    for batch in dev_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        labels = batch[1].to(device)

     
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)

        # Collect true labels and predictions
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())


    accuracy = 100 * sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
    f1 = f1_score(y_true, y_pred, average='macro')
    class_report = classification_report(y_true, y_pred)


    print(f"Accuracy: {accuracy:.2f}%")
    print(f"F1-score: {f1:.4f}")
    print("Classification Report:")
    print(class_report)


Accuracy: 26.06%
F1-score: 0.2067
Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       210
           1       0.26      1.00      0.41        74

    accuracy                           0.26       284
   macro avg       0.13      0.50      0.21       284
weighted avg       0.07      0.26      0.11       284



In [14]:
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')

test['transcriptions'] = test['transcriptions'].apply(text_preprocessing)
test

,image_id,transcriptions,image_data
0,335,ம்ஏ சௌஸல்ல்ய் இக்நோரிந்க் wஹேந் ஸோமேஓநே சோம்ப்...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
1,1719,பணத்தின் தத்துவம் வாழ்வதற்குச் செலவு மிகக்குறை...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."
2,624,ம்ஏ தகிந்க் போதோஸ் ஓஃப் ஏவேர்யோநே டேம் Vஅஅ வந்...,"[[[22, 28, 36], [22, 28, 31], [22, 28, 31], [2..."
3,317,ஷ்மித் ஆந்நந் பித்ச் ல பதுதுதேந் சுப் வந்கிது ...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
4,1576,ம்அநோ Pஹோதோங்ரித் ஈரு'ஆள் Bஸ்ரு பாத்துக்கிட்டே...,"[[[115, 80, 63], [115, 80, 63], [115, 80, 63],..."
...,...,...,...
351,434,ம்உது Kஉத்தி ஞுஸ்த் நோw ஈவலோ பேந்திந்க் தஸ்க் ...,"[[[52, 52, 52], [52, 52, 52], [52, 52, 52], [5..."
352,1227,காலையில வச்சேன் மாமியார் இன்னைக்கு வெள்ளி கிழம...,"[[[22, 22, 22], [22, 22, 22], [22, 22, 22], [2..."
353,879,ஈஸ்த் இந் ஸ்சோஓல்ல்சோல்லேகே ம்மே நேwஃப்ரிஏந்த்...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
354,602,ம்உது Kஉத்தி ஞுஸ்த் நோw ம்அநகேர் ட்ள் ரு'ஐஸே ப...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."


In [15]:
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import torch
import numpy as np


class TestDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']

    
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')

        return encoding


test_dataset = TestDataset(test, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16)


model.eval()


test_predictions = []

with torch.no_grad():
    for batch in test_loader:
      
        input_ids = batch['input_ids'].squeeze(1).to(device)
        attention_mask = batch['attention_mask'].squeeze(1).to(device)

      
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)

       
        test_predictions.extend(predicted.cpu().numpy())


test_predictions = np.array(test_predictions)


test['predictions'] = test_predictions

predictions_df = pd.DataFrame({
    'image_id': test['image_id'],
    'predictions': test_predictions
})

predictions_df.to_csv('mBERT Predictions.csv', index=False)
print("Predictions saved")


Predictions saved


# IndicBert

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim


class TextDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']
        label = self.df.loc[idx, 'labels']


        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')

        return encoding, torch.tensor(label)


tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
indicbert_model = AutoModel.from_pretrained('ai4bharat/indic-bert')


class TextClassifier(nn.Module):
    def __init__(self, indicbert_model, num_classes):
        super(TextClassifier, self).__init__()
        self.indicbert_model = indicbert_model
        self.fc = nn.Linear(768, num_classes)  # IndicBERT also has a hidden size of 768

    def forward(self, input_ids, attention_mask):
   
        bert_output = self.indicbert_model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        bert_pool = torch.mean(bert_output, 1)  # Mean pooling over token embeddings
        output = self.fc(bert_pool)
        return output


train_dataset = TextDataset(train, tokenizer)
dev_dataset = TextDataset(dev, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16)


num_classes = len(train['labels'].unique())
model = TextClassifier(indicbert_model, num_classes)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

for epoch in range(20):  # Number of epochs
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        labels = batch[1].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Evaluation
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for batch in dev_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        labels = batch[1].to(device)

        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}%")


In [ ]:
from sklearn.metrics import classification_report, f1_score
import numpy as np


model.eval()
with torch.no_grad():
    y_true = []
    y_pred = []
    
    for batch in dev_loader:
        input_ids = batch[0]['input_ids'].squeeze(1).to(device)
        attention_mask = batch[0]['attention_mask'].squeeze(1).to(device)
        labels = batch[1].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)

        # Collect true labels and predictions
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

    
    accuracy = 100 * sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
    f1 = f1_score(y_true, y_pred, average='macro')
    class_report = classification_report(y_true, y_pred)

    # Display results
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"F1-score: {f1:.4f}")
    print("Classification Report:")
    print(class_report)


In [ ]:
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')

test['transcriptions'] = test['transcriptions'].apply(text_preprocessing)
test

In [ ]:
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import torch
import numpy as np


class TestDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, 'transcriptions']

        # Tokenize text
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')

        return encoding


test_dataset = TestDataset(test, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16)


model.eval()


test_predictions = []

with torch.no_grad():
    for batch in test_loader:
     
        input_ids = batch['input_ids'].squeeze(1).to(device)
        attention_mask = batch['attention_mask'].squeeze(1).to(device)

   
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)


        test_predictions.extend(predicted.cpu().numpy())


test_predictions = np.array(test_predictions)

test['predictions'] = test_predictions

predictions_df = pd.DataFrame({
    'image_id': test['image_id'],
    'predictions': test_predictions
})

predictions_df.to_csv('IndicBert Predictions.csv', index=False)
print("Predictions saved")
